# Membrane Causal SPINN Forward Problem (Demo)

Simply-supported square membrane (2D wave equation), 1st eigenmode.

PDE: $u_{tt} = u_{xx} + u_{yy}$, domain $(x, y) \in [0,1]^2$, $t \in [0,1]$.

Three body networks (one per axis $t$, $x$, $y$), tensor-product merge.


In [ ]:
import jax
jax.config.update('jax_default_matmul_precision', 'float32')

import jax.numpy as jnp
import numpy as np
import optax
from jax import jvp, value_and_grad
from flax import linen as nn
from typing import Sequence
from functools import partial
from tqdm.auto import trange
import matplotlib.pyplot as plt


## Hyperparameters

In [ ]:
SEED       = 2
NC         = 64                  # collocation per axis  (64³ ≈ 262k points)
NC_TEST    = 100
LR         = 1e-5
EPOCHS     = 3_000               # bump to 100_000 for paper-quality results
N_LAYERS   = 4
FEATURES   = 128
R          = 128
T_MAX      = 1.0


## SPINN model + HVP

In [ ]:
# Forward-over-forward HVP.  Used to compute u_xx, u_tt, u_xxxx, etc.
def hvp_fwdfwd(f, primals, tangents, return_primals=False):
    g = lambda primals: jvp(f, (primals,), tangents)[1]
    primals_out, tangents_out = jvp(g, primals, tangents)
    if return_primals:
        return primals_out, tangents_out
    return tangents_out


In [ ]:
# Separable PINN with 3 axes (t, x, y).  Returns either a single (T, Nx, Ny)
# tensor or a list of `out_dim` tensors depending on out_dim.
class SPINN3d(nn.Module):
    features: Sequence[int]
    r: int
    out_dim: int = 1
    mlp: str = 'modified_mlp'

    @nn.compact
    def __call__(self, t, x, y):
        inputs, outputs, preds = [t, x, y], [], []
        init = nn.initializers.glorot_normal()
        for X in inputs:
            if self.mlp == 'mlp':
                for fs in self.features[:-1]:
                    X = nn.tanh(nn.Dense(fs, kernel_init=init)(X))
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(X)
            else:
                U = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                V = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                H = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                for fs in self.features[:-1]:
                    Z = nn.tanh(nn.Dense(fs, kernel_init=init)(H))
                    H = (1 - Z) * U + Z * V
                X = nn.Dense(self.r * self.out_dim, kernel_init=init)(H)
            outputs.append(jnp.transpose(X, (1, 0)))
        for i in range(self.out_dim):
            a = outputs[0][self.r * i:self.r * (i + 1)]
            b = outputs[1][self.r * i:self.r * (i + 1)]
            c = outputs[2][self.r * i:self.r * (i + 1)]
            xy = jnp.einsum('rt,rx->rtx', a, b)
            preds.append(jnp.einsum('rtx,ry->txy', xy, c))
        return preds if self.out_dim > 1 else preds[0]


## Analytic solution & data generator

In [ ]:
def exact_u(t, x, y):
    return jnp.sin(jnp.pi * x) * jnp.sin(jnp.pi * y) * jnp.cos(jnp.sqrt(2) * jnp.pi * t)

def source_term(t, x, y):
    return 0.0

def make_train_data(nc, key):
    keys = jax.random.split(key, 3)
    tc = jnp.linspace(0, T_MAX, nc + 2)[1:-1].reshape(-1, 1)
    xc = jnp.linspace(0, 1, nc + 2)[1:-1].reshape(-1, 1)
    yc = jnp.linspace(0, 1, nc + 2)[1:-1].reshape(-1, 1)
    tm, xm, ym = jnp.meshgrid(tc.ravel(), xc.ravel(), yc.ravel(), indexing='ij')
    uc = jnp.broadcast_to(source_term(tm, xm, ym), tm.shape)

    ti = jnp.zeros((nc, 1)); xi = xc; yi = yc
    ti_m, xi_m, yi_m = jnp.meshgrid(ti.ravel(), xi.ravel(), yi.ravel(), indexing='ij')
    ui = exact_u(ti_m, xi_m, yi_m)

    # Random boundary samples on the four walls of the square
    tb = jax.random.uniform(keys[0], (nc, 1), minval=0., maxval=T_MAX)
    xb = jax.random.uniform(keys[1], (nc, 1), minval=0., maxval=1.)
    yb = jax.random.uniform(keys[2], (nc, 1), minval=0., maxval=1.)
    xbl = jnp.zeros((nc, 1));  xbr = jnp.ones((nc, 1))
    ybup = jnp.ones((nc, 1));  ybdown = jnp.zeros((nc, 1))

    tbm_l, xbl_m, yb1 = jnp.meshgrid(tb.ravel(), xbl.ravel(), yb.ravel(), indexing='ij')
    tbm_r, xbr_m, yb2 = jnp.meshgrid(tb.ravel(), xbr.ravel(), yb.ravel(), indexing='ij')
    ubl = exact_u(tbm_l, xbl_m, yb1); ubr = exact_u(tbm_r, xbr_m, yb2)
    tbm_u, xb_m1, ybup1 = jnp.meshgrid(tb.ravel(), xb.ravel(), ybup.ravel(), indexing='ij')
    tbm_d, xb_m2, ybd1  = jnp.meshgrid(tb.ravel(), xb.ravel(), ybdown.ravel(), indexing='ij')
    ubup = exact_u(tbm_u, xb_m1, ybup1); ubdown = exact_u(tbm_d, xb_m2, ybd1)

    W = jnp.tril(jnp.ones((nc, nc)), k=-1)
    return (tc, xc, yc, uc, ti, xi, yi, ui,
            tb, xb, yb, xbl, xbr, ybup, ybdown,
            ubl, ubr, ubup, ubdown, W)


## Causal loss (3D residual)

In [ ]:
@partial(jax.jit, static_argnames=('apply_fn',))
def loss_and_grad(apply_fn, params, *train_data):
    (tc, xc, yc, uc, ti, xi, yi, ui,
     tb, xb, yb, xbl, xbr, ybup, ybdown,
     ubl, ubr, ubup, ubdown, W) = train_data

    def residual_loss(p):
        u = apply_fn(p, tc, xc, yc)
        v = jnp.ones(tc.shape)
        utt = hvp_fwdfwd(lambda t: apply_fn(p, t, xc, yc), (tc,), (v,))
        uxx = hvp_fwdfwd(lambda x: apply_fn(p, tc, x, yc), (xc,), (v,))
        uyy = hvp_fwdfwd(lambda y: apply_fn(p, tc, xc, y), (yc,), (v,))
        res = uxx + uyy - uc - utt
        loss_time = jnp.mean(res**2, axis=(1, 2), keepdims=True).reshape(-1)
        agg = jax.lax.stop_gradient(jnp.dot(W, loss_time))
        causal_w = jnp.exp(-3.0 * agg)
        return jnp.mean(causal_w * loss_time)

    def initial_loss(p):
        ic = jnp.mean((apply_fn(p, ti, xi, yi) - ui)**2)
        v_t = jnp.ones(ti.shape)
        u_t0 = jvp(lambda t: apply_fn(p, t, xi, yi), (ti,), (v_t,))[1]
        return ic + jnp.mean(u_t0**2)

    def boundary_loss(p):
        ul = apply_fn(p, tb, xbl, yb);  ur = apply_fn(p, tb, xbr, yb)
        uu = apply_fn(p, tb, xb, ybup); ud = apply_fn(p, tb, xb, ybdown)
        return (jnp.mean((ul - ubl)**2) + jnp.mean((ur - ubr)**2)
              + jnp.mean((uu - ubup)**2) + jnp.mean((ud - ubdown)**2))

    total = lambda p: 0.01 * residual_loss(p) + initial_loss(p) + boundary_loss(p)
    return value_and_grad(total)(params)


## Initialize and train

In [ ]:
key = jax.random.PRNGKey(SEED)
key, sub_init, sub_data = jax.random.split(key, 3)

model = SPINN3d(features=[FEATURES] * N_LAYERS, r=R, out_dim=1)
t0 = jnp.ones((NC, 1)); x0 = jnp.ones((NC, 1)); y0 = jnp.ones((NC, 1))
params = model.init(sub_init, t0, x0, y0)
apply_fn = jax.jit(model.apply)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total trainable params: {n_params}")

train_data = make_train_data(NC, sub_data)

# Eval grid + ground truth (for periodic best-tracking)
t_eval = jnp.linspace(0, T_MAX, NC_TEST).reshape(-1, 1)
x_eval = jnp.linspace(0, 1, NC_TEST).reshape(-1, 1)
y_eval = jnp.linspace(0, 1, NC_TEST).reshape(-1, 1)
tm_eval, xm_eval, ym_eval = jnp.meshgrid(t_eval.ravel(), x_eval.ravel(), y_eval.ravel(), indexing='ij')
u_true_eval = exact_u(tm_eval, xm_eval, ym_eval)

optim = optax.adam(LR); state = optim.init(params)

losses = []
best_loss = 1e9
best_err = 1e9
best_params = params

pbar = trange(EPOCHS)
for e in pbar:
    loss, grads = loss_and_grad(apply_fn, params, *train_data)
    updates, state = optim.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    losses.append(float(loss))
    if float(loss) <= best_loss:                       
        best_loss = float(loss)
        u_pred_eval = apply_fn(params, t_eval, x_eval, y_eval)
        best_err = float(jnp.linalg.norm(u_pred_eval - u_true_eval) / jnp.linalg.norm(u_true_eval))
        best_params = params
    if (e + 1) % 500 == 0:
        pbar.set_postfix(loss=f"{loss:.3e}", err_at_best_loss=f"{best_err:.3e}")

print(f"\nRel L2 error at best-loss step (full domain): {best_err:.3e}")


## Evaluate and plot  params at best-loss (slice at t = 1/3)

In [ ]:
# Relative L2 error against the analytic solution.
def relative_l2(pred, true):
    return float(jnp.linalg.norm(pred - true) / jnp.linalg.norm(true))

# Full 3D evaluation grid (using BEST-tracked params)
t_test = jnp.linspace(0, T_MAX, NC_TEST).reshape(-1, 1)
x_test = jnp.linspace(0, 1, NC_TEST).reshape(-1, 1)
y_test = jnp.linspace(0, 1, NC_TEST).reshape(-1, 1)
tm, xm, ym = jnp.meshgrid(t_test.ravel(), x_test.ravel(), y_test.ravel(), indexing='ij')
u_true_all = exact_u(tm, xm, ym)
u_pred_all = apply_fn(best_params, t_test, x_test, y_test)
err = best_err
print(f"Best relative L2 error (full domain): {err:.3e}")

# Slice plot at t* ≈ 1/3
idx = NC_TEST // 3
xy_x, xy_y = np.meshgrid(np.asarray(x_test).ravel(),
                         np.asarray(y_test).ravel(), indexing='ij')
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
data = [u_true_all[idx], u_pred_all[idx], jnp.abs(u_true_all[idx] - u_pred_all[idx])]
titles = ['Exact $u$', 'Predicted $\hat u$', '|Error|']
for ax, d, ti_ in zip(axs, data, titles):
    im = ax.pcolormesh(xy_x, xy_y, np.asarray(d), cmap='RdBu_r', shading='auto')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f"{ti_}  (t={float(t_test[idx, 0]):.3f})")
    plt.colorbar(im, ax=ax); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(losses); plt.xlabel('epoch'); plt.ylabel('total loss')
plt.title('Membrane Causal SPINN — training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
